In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# ==========================================================
# Master Claim Status Dimension
# ==========================================================

status_data = [

    # ---------------- APPROVED ----------------

    ("APPROVED","Fully Paid","Approved","Low"),
    ("APPROVED","Fully Paid","Approved","Medium"),
    ("APPROVED","Fully Paid","Approved","High"),

    ("APPROVED","Partially Paid","Approved","Low"),
    ("APPROVED","Partially Paid","Approved","Medium"),
    ("APPROVED","Partially Paid","Approved","High"),

    # ---------------- DENIED ----------------

    ("DENIED","Unpaid","No Insurance Payment","Low"),
    ("DENIED","Unpaid","No Insurance Payment","Medium"),
    ("DENIED","Unpaid","No Insurance Payment","High"),

    ("DENIED","Unpaid","Invalid Claim Amount","Low"),
    ("DENIED","Unpaid","Invalid Claim Amount","Medium"),
    ("DENIED","Unpaid","Invalid Claim Amount","High"),

    ("DENIED","Unpaid","Payment Not Available","Low"),
    ("DENIED","Unpaid","Payment Not Available","Medium"),
    ("DENIED","Unpaid","Payment Not Available","High"),

    # ---------------- PENDING ----------------

    ("PENDING REVIEW","Unknown","Claim Amount Missing","Low"),
    ("PENDING REVIEW","Unknown","Claim Amount Missing","Medium"),
    ("PENDING REVIEW","Unknown","Claim Amount Missing","High"),

    ("PENDING REVIEW","Unknown","Payment Not Available","Low"),
    ("PENDING REVIEW","Unknown","Payment Not Available","Medium"),
    ("PENDING REVIEW","Unknown","Payment Not Available","High")

]

# ==========================================================
# Create DataFrame
# ==========================================================

dim_claim_status = spark.createDataFrame(
    status_data,
    [
        "CLAIM_STATUS",
        "PAYMENT_STATUS",
        "DENIAL_REASON",
        "RISK_LEVEL"
    ]
)

# ==========================================================
# Generate Surrogate Key
# ==========================================================

window_spec = Window.orderBy(
    "CLAIM_STATUS",
    "PAYMENT_STATUS",
    "DENIAL_REASON",
    "RISK_LEVEL"
)

dim_claim_status = (
    dim_claim_status
    .withColumn(
        "STATUS_KEY",
        row_number().over(window_spec)
    )
)

# ==========================================================
# Audit Column
# ==========================================================

dim_claim_status = (
    dim_claim_status
    .withColumn(
        "GOLD_CREATED_TIMESTAMP",
        current_timestamp()
    )
)

# ==========================================================
# Reorder Columns
# ==========================================================

dim_claim_status = dim_claim_status.select(

    "STATUS_KEY",
    "CLAIM_STATUS",
    "PAYMENT_STATUS",
    "DENIAL_REASON",
    "RISK_LEVEL",
    "GOLD_CREATED_TIMESTAMP"

)

# ==========================================================
# Write Gold Table
# ==========================================================

(
    dim_claim_status.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(
        "healthcare_claims_catalog.gold.dim_claim_status"
    )
)

# ==========================================================
# Validation
# ==========================================================

print("="*60)
print("Gold Dimension - Claim Status Created Successfully")
print("="*60)

print("Total Status Records :", dim_claim_status.count())

dim_claim_status.printSchema()

dim_claim_status.show(50, False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold Dimension - Claim Status Created Successfully
Total Status Records : 21
root
 |-- STATUS_KEY: integer (nullable = false)
 |-- CLAIM_STATUS: string (nullable = true)
 |-- PAYMENT_STATUS: string (nullable = true)
 |-- DENIAL_REASON: string (nullable = true)
 |-- RISK_LEVEL: string (nullable = true)
 |-- GOLD_CREATED_TIMESTAMP: timestamp (nullable = false)

+----------+--------------+--------------+---------------------+----------+--------------------------+
|STATUS_KEY|CLAIM_STATUS  |PAYMENT_STATUS|DENIAL_REASON        |RISK_LEVEL|GOLD_CREATED_TIMESTAMP    |
+----------+--------------+--------------+---------------------+----------+--------------------------+
|1         |APPROVED      |Fully Paid    |Approved             |High      |2026-08-06 17:55:55.706871|
|2         |APPROVED      |Fully Paid    |Approved             |Low       |2026-08-06 17:55:55.706871|
|3         |APPROVED      |Fully Paid    |Approved             |Medium    |2026-08-06 17:55:55.706871|
|4         |APPROVED